# Smart Buildings — Version 6: Online Learning and Action Planning

**Experiment:** Online thermal response estimation and action-conditioned optimization. This is distinct from V2/V3 PID, V4 passive trend forecasting, and V5 heuristic comfort-risk allocation. **V6 has not been scored on Kaggle. Lower scores are better.**

## 1. One five-minute decision

~~~text
Building metadata + live zone/AHU sensors
                  |
        Read zone temperatures and setpoints
                  |
   Update each zone's model from the PREVIOUS action
                  |
  Enumerate seven AHU modes (off / vent / cooling / heating / mixed)
                  |
          Remove unsafe AHU choices
                  |
     For each AHU option, search damper/reheat
                  |
   Predict comfort at 20, 40, and 60 minutes
       plus approximate energy expenditure
                  |
           Select minimum estimated cost
                  |
       Issue AHU + VAV control settings
                  |
        Coordinate shared hot-water plant
                  |
         Remember current action features
                  |
            Return Kaggle action arrays
~~~

**Physical constraint:** AHUs can cool but cannot directly heat. Every zone has a VAV damper and reheat coil; the reheat coils share a central hot-water system. One AHU serves several zones with one supply-air temperature.

## 2. Component reference

| Component | Python method or class | Responsibility |
| --- | --- | --- |
| Building topology | TransferPolicy.__init__ | Map arbitrary building-specific sensors, AHUs, VAVs, and hot-water actions to flat array positions. |
| Thermal feature engineering | ZoneResponseModel.features | Convert outdoor-vs-room temperature and issued cooling/heating actions into four model features. |
| Online system identification | ZoneResponseModel.observe, predict_delta, remember | Learn four bounded RLS coefficients per zone from the previous action and measured five-minute temperature change, then predict effects of candidate actions. |
| Episode reset | TransferPolicy.reset, ZoneResponseModel.clear_pending | Reset learned state between scenarios, and discard broken transitions caused by missing sensors. |
| Zone observation | _fallback_band, _read_zones | Denormalize zone temperatures, read current heating/cooling setpoints and use the occupied-hour fallback if needed. |
| Candidate configurations | OPERATING_POINTS | Define seven AHU options: off, ventilate, light/strong cool, light/strong reheat, or mixed. |
| Zone prediction/scoring | _violation, _score_vav | Score predicted future comfort loss and marginal VAV energy using a simple model. |
| Per-zone search | _best_vav | Find a low-cost damper/reheat command on a 4-by-3 discrete candidate grid. |
| Safety guardrails | _candidate_allowed | Reject full-off and single-sided inappropriate modes when zone deviations are severe. |
| AHU joint optimization | _select_operating_point, _plant_cost | Evaluate remaining AHU candidates, each with optimized VAV choices, and add approximate plant energy cost. |
| AHU equipment outputs | _apply_ahu_action, _economizer | Command run state, supply-air temperature, fan pressure and outdoor-air/return-air mixing. |
| Per-zone equipment outputs | _apply_vav_actions | Set VAV damper and reheat valve, and store the issued-action features needed at the next observation. |
| Shared hot water | _apply_shared_hot_water | Combine demands across all AHUs, enable pump if needed and set water supply temperature/pressure. |
| Main orchestration | _initial_actions, get_action | Initialize safe defaults, run the complete learn-plan-act pipeline and return finite, typed, normalized arrays. |

**Recommended reading order:** Start with TransferPolicy.get_action for the control flow; then _select_operating_point and _best_vav for optimization; finally ZoneResponseModel for how thermal dynamics are learned.

## 3. What makes the model action-conditioned?

If an earlier command opened a cooling damper and five minutes later the zone became cooler, ZoneResponseModel.observe updates the cooling-response coefficient from the *previous action's model features* and the measured temperature change. When planning the next step, predict_delta estimates how different damper, supply-air and reheat choices would change temperature.

This differs from passively extrapolating the most recent temperature slope. Before enough observations exist, the model uses fixed, physically direction-constrained prior coefficients.

## 4. Units, assumptions, and limitations

- Physical temperatures use **Kelvin**. Zone temperature readings are normalized; zone heating/cooling setpoint sensors are raw Kelvin.
- Continuous commands are float32 within **[-1, 1]**, and discrete run commands are int32 in **0 or 1**. A normalized continuous zero is the **physical midpoint**, not equipment OFF.
- The online model uses bounded recursive least squares with a 0.995 forgetting factor; it is a simplified action-response model rather than complete building physics.
- Predictions at 20/40/60 minutes extrapolate a one-step model under approximately constant conditions. This is **not** a simulator-backed multi-step MPC rollout.
- Comfort, fan and heating costs are heuristic surrogates, **not the exact Kaggle reward**. Per-zone occupancy weights are not available.
- Set LEARN_DYNAMICS to false to test a fixed model, or OPTIMIZE_ENERGY to false for comfort-only optimization. Candidate modes and coefficients are not externally trained.
- No Internet access, pretrained weights, fixed building identifiers, or hidden weather inputs are required.

## 5. Validation and submission

Synthetic tests exercise the exact notebook policy code with variable building layouts, missing sensors, cooling, heating, mixed scenarios, and model resets. A separate differential test compares the original and refactored controller across randomized inputs, including learning state. These establish interface behavior, not leaderboard performance.

The final notebook cell runs the official SmartBuildingsInferenceServer(TransferPolicy). Full 30-day public simulator evaluation and Kaggle hidden scoring remain outstanding.


In [ ]:
"""Version 6: online thermal system identification + finite-action planning.

Unlike Versions 2/3 (PID), 4 (trend extrapolation) and 5 (rule-based
risk allocation with one adaptive gain), this policy learns a small
ACTION-CONDITIONED response model for each zone. At every five-minute
step, it scores a bounded collection of HVAC configurations, including
individual VAV dampers and valves, over a 60-minute planning horizon.

The learned model is deliberately small and regularized. Its inputs are
physical temperature differences and the PREVIOUS actions issued to the
building. No saved weights, data files, external network, or fixed building
size are required. The fallback model operates before online data arrive.
"""
import re
import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class ZoneResponseModel:
    """Fit one zone's five-minute Kelvin change using bounded online RLS.

    Regression: ΔT = b0 * ((Tout-T)/20) + b1 * cooling + b2 * reheat + b3.
    The features encode *issued actions*, so the model can compare the
    consequences of candidate commands, unlike a passive trend estimator.
    Coefficient signs are constrained by physical direction; the priors are
    used from step one and the covariance limits the size of updates.
    """

    def __init__(self):
        self.weights = np.array([0.035, -0.16, 0.20, 0.0], dtype=np.float64)
        self.covariance = np.diag([0.45, 0.55, 0.55, 0.12])
        self.last_temperature = None
        self.last_features = None
        self.samples = 0

    @staticmethod
    def features(temp, outdoor, run, supply, damper, valve, water_on, water_temp):
        """Produce action-dependent ambient/cooling/reheat/bias features.

        Temperatures use Kelvin; damper/valve are fractions, not commands.
        """
        outdoor = outdoor if np.isfinite(outdoor) else temp
        ambient = np.clip((outdoor - temp) / 20.0, -3.0, 3.0)
        # AHU cooling coil cannot heat. Natural outdoor mixing can cool air
        # further than the supply setpoint even with the chiller inactive.
        mixed_air = 0.70 * temp + 0.30 * outdoor
        delivered = min(supply, mixed_air)
        cooling = run * damper * np.clip((temp - delivered) / 12.0, 0.0, 2.0)
        reheat = (run * damper * valve * water_on *
                  np.clip((water_temp - temp) / 55.0, 0.0, 2.0))
        return np.array([ambient, cooling, reheat, 1.0], dtype=np.float64)

    def observe(self, current_temperature):
        """Update RLS coefficients from the previous issued action's outcome.

        Forgetting factor 0.995; clip residual and physically directional
        coefficients; discard implausible >1.5K five-minute jumps.
        """
        if self.last_temperature is None or self.last_features is None:
            return
        delta = current_temperature - self.last_temperature
        if not np.isfinite(delta) or abs(delta) > 1.5:
            return  # sensor discontinuity, not ordinary thermal response
        x = self.last_features
        px = self.covariance @ x
        denominator = 0.995 + x @ px
        gain = px / denominator
        predicted = float(self.weights @ x)
        self.weights += gain * np.clip(delta - predicted, -0.8, 0.8)
        self.weights = np.clip(self.weights,
                               [0.002, -0.65, 0.004, -0.12],
                               [0.20, -0.004, 0.70, 0.12])
        self.covariance = (self.covariance - np.outer(gain, px)) / 0.995
        self.covariance = 0.5 * (self.covariance + self.covariance.T)
        self.samples += 1

    def predict_delta(self, features):
        return float(np.clip(self.weights @ features, -0.50, 0.50))

    def remember(self, temperature, features):
        self.last_temperature = float(temperature)
        self.last_features = features.copy()

    def clear_pending(self):
        self.last_temperature = None
        self.last_features = None


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Select an approximately economical action using online learned dynamics.

    At every step: update each zone's RLS model from the preceding action,
    enumerate a few physical AHU/boiler operating points, optimize each
    zone's VAV fractions for those points, and choose the lowest predicted
    60-minute discomfort + energy surrogate. Safety guardrails prevent the
    optimizer from leaving a severely uncomfortable building switched off.
    """

    # Ablations for follow-up evaluation, not fitted from hidden scenarios.
    LEARN_DYNAMICS = True
    OPTIMIZE_ENERGY = True
    PLAN_STEPS = (4, 8, 12)   # 20, 40 and 60 minutes
    DISCOMFORT_WEIGHT = 1.0
    ENERGY_WEIGHT = 1.0
    # Candidate tuple: (mode_name, AHU_running, supply_air_K, fan_pressure_Pa,
    #                   hot_water_K, hot_water_enabled_boolean).
    OPERATING_POINTS = (
        ('off', False, 305., 200., 310., False),
        ('ventilate', True, 305., 200., 310., False),
        ('cool_light', True, 294., 320., 310., False),
        ('cool_strong', True, 286., 900., 310., False),
        ('heat_light', True, 305., 360., 332., True),
        ('heat_strong', True, 305., 920., 348., True),
        ('mixed', True, 290., 600., 338., True),
    )

    def __init__(self, building_meta_data, action_spec):
        """Map variable-sized building sensors and actuator locations from metadata."""
        super().__init__(building_meta_data, action_spec)
        m = building_meta_data
        self.continuous_count = int(m['n_continuous_action'])
        self.discrete_count = int(m['n_discrete_action'])
        self.fields = {name: i for i, name in enumerate(m['observation_field_names'])}
        self.ahu_supply_actions = [int(x) for x in m['temp_indices']]
        self.ahu_pressure_actions = [int(x) for x in m['pressure_indices']]
        self.ahu_recirc_actions = [int(x) for x in m['recirculation_indices']]
        self.water_actions = [int(x) for x in m['water_indices']]
        self.water_temp_action = self.water_actions[0] if self.water_actions else None
        n_ahu = len(self.ahu_supply_actions)
        self.water_pressure_action = (self.ahu_pressure_actions[n_ahu]
                                      if len(self.ahu_pressure_actions) > n_ahu else None)
        self.water_run_action = self.discrete_count - 1
        self.comfort = tuple(map(float, m['comfort_temp_window']))
        self.eco = tuple(map(float, m['eco_temp_window']))
        self.morning = float(m['morning_start_hour'])
        self.evening = float(m['evening_start_hour'])
        temp_normalizer = m.get('observation_normalization_metadata', {}).get(
            'zone_air_temperature_sensor', {})
        self.temp_mean = float(temp_normalizer.get('sample_mean', 0.0))
        self.temp_std = float(temp_normalizer.get('sample_std', 1.0))
        if not np.isfinite(self.temp_std) or self.temp_std <= 0:
            self.temp_std = 1.0

        def numeric_order(value):
            match = re.search(r'\d+', str(value))
            return (int(match.group()) if match else -1, str(value))

        self.ahus = []
        for number, ahu_id in enumerate(sorted(m['ahu_zone_assignments'], key=numeric_order)[:n_ahu]):
            zones = []
            for room in m['ahu_zone_assignments'][ahu_id]:
                vav = str(room).replace('room_', 'VAV_')
                sensor = self.fields.get(vav + '_zone_air_temperature_sensor')
                if sensor is None:
                    continue
                actions = m['zone_action_indices'].get(vav, {})
                zones.append({
                    'name': vav,
                    'sensor': sensor,
                    'heat_setpoint': self.fields.get(vav + '_zone_air_heating_temperature_setpoint_sensor'),
                    'cool_setpoint': self.fields.get(vav + '_zone_air_cooling_temperature_setpoint_sensor'),
                    'damper_action': actions.get('supply_air_damper_percentage_command'),
                    'valve_action': actions.get('reheat_valve_percentage_command'),
                    'model': ZoneResponseModel(),
                })
            self.ahus.append({
                'zones': zones, 'run_action': number,
                'supply_action': self.ahu_supply_actions[number],
                'pressure_action': self.ahu_pressure_actions[number] if number < len(self.ahu_pressure_actions) else None,
                'recirc_action': self.ahu_recirc_actions[number] if number < len(self.ahu_recirc_actions) else None,
                'outside_sensor': self.fields.get('ahs_' + str(ahu_id) + '_outside_air_temperature_sensor'),
            })
        self.reset()

    def reset(self):
        """Isolate episodes, including online model state and pending actions."""
        for ahu in self.ahus:
            for zone in ahu['zones']:
                zone['model'] = ZoneResponseModel()

    @staticmethod
    def _command(fraction):
        """Map physical actuator fraction [0, 1] to normalized [-1, 1]."""
        return float(np.clip(2.0 * fraction - 1.0, -1.0, 1.0))

    @staticmethod
    def _supply_command(kelvin):
        """Map 285..305 Kelvin supply-air setpoint into [-1, 1]."""
        return float(np.clip((kelvin - 285.0) / 10.0 - 1.0, -1.0, 1.0))

    def _fallback_band(self, obs):
        """Use weekday/hour only if current zone setpoint sensors are invalid."""
        def cyclic(sine, cosine, period):
            a, b = self.fields.get(sine), self.fields.get(cosine)
            if a is None or b is None:
                return None
            if not np.isfinite(obs[a]) or not np.isfinite(obs[b]):
                return None
            return (np.arctan2(obs[a], obs[b]) % (2.0 * np.pi)) * period / (2.0 * np.pi)
        hour = cyclic('hod_sin_000', 'hod_cos_000', 24)
        day = cyclic('dow_sin_000', 'dow_cos_000', 7)
        occupied = (hour is not None and day is not None and day < 5 and
                    self.morning <= hour < self.evening)
        return self.comfort if occupied else self.eco

    def _read_zones(self, obs, ahu, fallback):
        """Denormalize zone temperatures, validate Kelvin setpoints, and learn.

        Observe the temperature response to the PREVIOUS issued action before
        selecting today's candidate. Invalid sensors clear pending transitions.
        """
        result = []
        outdoor_idx = ahu['outside_sensor']
        outdoor = float(obs[outdoor_idx]) if outdoor_idx is not None else np.nan
        for zone in ahu['zones']:
            temp = float(obs[zone['sensor']]) * self.temp_std + self.temp_mean
            if not np.isfinite(temp) or not 240.0 <= temp <= 340.0:
                zone['model'].clear_pending()
                continue
            li, ui = zone['heat_setpoint'], zone['cool_setpoint']
            low = float(obs[li]) if li is not None else np.nan
            high = float(obs[ui]) if ui is not None else np.nan
            if not (np.isfinite(low) and np.isfinite(high) and
                    245.0 <= low < high <= 335.0 and high-low <= 35.0):
                low, high = fallback
            if self.LEARN_DYNAMICS:
                zone['model'].observe(temp)
            result.append({'zone': zone, 'temp': temp, 'low': low,
                           'high': high, 'outdoor': outdoor})
        return result

    @staticmethod
    def _violation(temp, low, high):
        """Comfort deviation in K, with convex penalty beyond the 5K knee."""
        outside = max(low - temp, temp - high, 0.0)
        return outside / 5.0 if outside <= 5.0 else 1.0 + ((outside - 5.0) / 5.0)**2

    def _score_vav(self, state, config, damper, valve):
        """Score one candidate VAV setting using learned action responses.

        Predict a five-minute temperature change, project to 20/40/60 minutes
        and average discomfort. Optionally add a marginal energy cost proxy.
        This is not the physical simulator or the official reward function.
        """
        name, running, supply, pressure, water_temp, water_on = config
        model = state['zone']['model']
        x = model.features(state['temp'], state['outdoor'], float(running),
                           supply, damper, valve, float(water_on), water_temp)
        delta = model.predict_delta(x)
        comfort = np.mean([self._violation(state['temp'] + delta * steps,
                                          state['low'], state['high'])
                           for steps in self.PLAN_STEPS])
        if not self.OPTIMIZE_ENERGY:
            return float(self.DISCOMFORT_WEIGHT * comfort)
        # Small marginal energy charges for increased damper + heating load.
        actuator_cost = ((0.012 * damper if running else 0.0) +
                         (0.022 * damper * valve if water_on else 0.0))
        return float(self.DISCOMFORT_WEIGHT * comfort +
                     self.ENERGY_WEIGHT * actuator_cost)

    def _best_vav(self, state, config):
        """Try up to four damper and three reheat fractions per VAV box.

        Returns (candidate_cost, damper_fraction, valve_fraction). The
        simulator enforces a 10% minimum damper opening on a running AHU.
        """
        _, running, _, _, _, water_on = config
        # Fractions (not normalized action units). Running AHUs maintain a
        # simulator-enforced 10% ventilation floor at each VAV.
        dampers = (0.1, 0.35, 0.7, 1.0) if running else (0.0,)
        valves = (0.0, 0.5, 1.0) if water_on else (0.0,)
        best = (np.inf, 0.0, 0.0)
        for damper in dampers:
            for valve in valves:
                cost = self._score_vav(state, config, damper, valve)
                if cost < best[0]:
                    best = (cost, damper, valve)
        return best

    @staticmethod
    def _plant_cost(config):
        """Estimate relative fan, cooling and boiler cost (not true kWh)."""
        _, running, supply, pressure, water_temp, water_on = config
        if not running:
            return 0.0
        fan = 0.035 + 0.035 * (pressure / 2000.0)**1.5
        cooling = 0.025 * max(0.0, 299.0 - supply) / 14.0
        heating = 0.025 * max(0.0, water_temp - 310.0) / 40.0 if water_on else 0.0
        return fan + cooling + heating

    @staticmethod
    def _candidate_allowed(config, worst_hot, worst_cold):
        """Reject unsafe AHU modes before scoring the learned predictions.

        When any zone is >=2 K outside the comfort band, staying completely
        off is forbidden. At >=4 K, drop operating modes that cannot address
        the extreme hot or cold condition. Mixed cooling/reheat remains.
        """
        name = config[0]
        if name == 'off' and max(worst_hot, worst_cold) >= 2.0:
            return False
        if worst_hot >= 4.0 and name in ('heat_light', 'heat_strong', 'ventilate'):
            return False
        if worst_cold >= 4.0 and name in ('cool_light', 'cool_strong', 'ventilate'):
            return False
        return True

    def _select_operating_point(self, zones):
        """Choose the lowest-cost joint AHU + VAV actions.

        First apply hard candidate restrictions. For each allowed AHU mode,
        independently optimize each VAV, average the per-zone model-based
        predicted discomfort and optionally add the AHU plant cost.
        Returns (operating_config, per_zone_choices).
        """
        worst_hot = max((z['temp'] - z['high'] for z in zones), default=0.0)
        worst_cold = max((z['low'] - z['temp'] for z in zones), default=0.0)
        best_cost, best_config, best_vavs = np.inf, None, None
        for config in self.OPERATING_POINTS:
            if not self._candidate_allowed(config, worst_hot, worst_cold):
                continue
            selections = [self._best_vav(state, config) for state in zones]
            score = sum(choice[0] for choice in selections) / len(zones)
            if self.OPTIMIZE_ENERGY:
                score += self.ENERGY_WEIGHT * self._plant_cost(config)
            if score < best_cost:
                best_cost, best_config, best_vavs = score, config, selections
        return best_config, best_vavs

    def _economizer(self, outdoor, zone_temperature, supply, pressure):
        """Choose recirculation when free cooling saves more than fan work.

        Returned +1 requests maximum recirculation, capped by the simulator.
        """
        if not np.isfinite(outdoor) or outdoor >= zone_temperature:
            return 1.0
        if zone_temperature - outdoor <= pressure * 0.00322:
            return 1.0
        fraction_outside = np.clip((zone_temperature - supply) /
                                   max(1e-6, zone_temperature-outdoor), 0.3, 1.0)
        return self._command(1.0 - fraction_outside)

    def _initial_actions(self):
        """Allocate safe idle commands, with correct array sizes and types.

        A normalized continuous zero is the MIDPOINT of a physical range,
        not OFF. +1 requests warmest AHU supply and maximum recirculation.
        """
        continuous = np.full(self.continuous_count, -1.0, dtype=np.float32)
        discrete = np.zeros(self.discrete_count, dtype=np.int32)
        for action_index in self.ahu_supply_actions + self.ahu_recirc_actions:
            continuous[action_index] = 1.0
        return continuous, discrete

    def _apply_ahu_action(self, ahu, states, config, continuous, discrete):
        """Command one AHU's run flag, cooling supply, fan and economizer.

        AHUs cannot heat: thermal reheat occurs downstream in the VAV boxes.
        """
        _, running, supply, pressure, water_temp, water_on = config
        if running:
            discrete[ahu['run_action']] = 1
            continuous[ahu['supply_action']] = self._supply_command(supply)
            if ahu['pressure_action'] is not None:
                continuous[ahu['pressure_action']] = self._command(pressure / 2000.0)
            if ahu['recirc_action'] is not None and supply < 300.0:
                outside = states[0]['outdoor']
                mean_zone = float(np.mean([s['temp'] for s in states]))
                continuous[ahu['recirc_action']] = self._economizer(
                    outside, mean_zone, supply, pressure)

    def _apply_vav_actions(self, states, config, choices, continuous, heat_demands):
        """Command per-zone airflow/reheat and store the issued model features.

        Per-zone choices are (estimated_cost, damper_fraction, valve_fraction).
        Missing actuators disable reheat or use the existing airflow proxy.
        The model observes each issued action's result on the NEXT timestep.
        """
        _, running, supply, pressure, water_temp, water_on = config
        for state, (_, damper, valve) in zip(states, choices):
            zone = state['zone']
            damper_index, valve_index = zone['damper_action'], zone['valve_action']
            if valve_index is None:
                valve = 0.0
            if damper_index is None:
                damper = 0.10 if running else 0.0
            if damper_index is not None:
                continuous[damper_index] = self._command(damper)
            if valve_index is not None:
                continuous[valve_index] = self._command(valve)
            if running and water_on and valve > 0.0:
                heat_demands.append((water_temp, pressure, damper*valve))
            features = zone['model'].features(
                state['temp'], state['outdoor'], float(running), supply,
                damper, valve, float(water_on and valve > 0.0), water_temp)
            zone['model'].remember(state['temp'], features)

    def _apply_shared_hot_water(self, heat_demands, continuous, discrete):
        """Combine reheat requests across AHUs into one building-wide plant.

        Enable the pump only for an actual reheat request, use the highest
        requested water temperature, and scale pressure to average load.
        """
        if heat_demands:
            discrete[self.water_run_action] = 1
            hot_water = max(demand[0] for demand in heat_demands)
            loading = np.mean([demand[2] for demand in heat_demands])
            if self.water_temp_action is not None:
                continuous[self.water_temp_action] = self._command((hot_water - 310.0) / 40.0)
            if self.water_pressure_action is not None:
                continuous[self.water_pressure_action] = self._command(max(0.35, loading))

    def get_action(self, obs):
        """One step: learn previous outcome -> plan -> issue -> remember.

        (1) Decode live sensors and update the online response models.
        (2) Search candidate AHU configurations and VAV settings.
        (3) Issue HVAC commands and remember model features for the next step.
        (4) Coordinate hot water and return normalized, typed action arrays.
        """
        obs = np.asarray(obs, dtype=np.float64).reshape(-1)
        continuous, discrete = self._initial_actions()
        fallback = self._fallback_band(obs)
        heat_demands = []

        for ahu in self.ahus:
            states = self._read_zones(obs, ahu, fallback)
            if not states:
                continue
            config, choices = self._select_operating_point(states)
            self._apply_ahu_action(ahu, states, config, continuous, discrete)
            self._apply_vav_actions(states, config, choices, continuous, heat_demands)

        self._apply_shared_hot_water(heat_demands, continuous, discrete)
        np.nan_to_num(continuous, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(continuous, -1.0, 1.0, out=continuous)
        return {'continuous_action': continuous, 'discrete_action': discrete}


In [ ]:
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
